In [ ]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from qaoa_rw import data, qubo, qaoa, baselines, metrics

RESULTS, FIGS = pathlib.Path("../results"), pathlib.Path("../figures")
RESULTS.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)
SEED = 0
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


In [ ]:
import glob
sim = pd.read_csv(RESULTS / "02_sim_summary.csv", index_col=0)
display(sim.round(4))

fig, ax = plt.subplots(figsize=(7, 3.8))
qs = sim[sim.index.str.startswith("QAOA")]
ax.bar(qs.index, qs["razón_aprox"], color="#2a6f97", label="QAOA (sin ruido)")
for name, c in [("Greedy", "#e07a5f"), ("Simulated annealing", "#81b29a")]:
    ax.axhline(sim.loc[name, "razón_aprox"], ls="--", color=c, label=name)
ax.axhline(1, color="k", lw=.8)
ax.set(ylim=(0, 1.05), ylabel="razón de aproximación", title="12 qubits · selección de variables"); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig(FIGS / "04_sim_summary.png", dpi=150); plt.show()


In [ ]:
prob_file = RESULTS / "hw_problem.npz"
if not prob_file.exists():
    print("Ejecuta primero el notebook 03.")
else:
    from qaoa_rw import ibm
    pb = np.load(prob_file); Qh, off_h, n, k = pb["Q"], float(pb["offset"]), int(pb["n"]), int(pb["k"])
    Eh = qubo.energies(Qh, off_h)
    ideal_params = {int(p): np.array(v) for p, v in json.load(open(RESULTS / "hw_ideal.json")).items()}

    rows = []
    for p, par in ideal_params.items():
        pr = qaoa.qaoa_probs(par, Eh, n)
        rows.append({"fuente": "Ideal", "p": p, **metrics.summarize(Eh, pr, n, k)})
    rows.append({"fuente": "Azar uniforme", "p": 0, **metrics.summarize(Eh, np.full(len(Eh), 1 / len(Eh)), n, k)})

    noisy_file = RESULTS / "hw_noisy_sim.json"
    if noisy_file.exists():
        for p, c in json.load(open(noisy_file)).items():
            rows.append({"fuente": "Simulador ruidoso", "p": int(p), **metrics.summarize(Eh, ibm.counts_to_prob(c, n), n, k)})
    for f in sorted(glob.glob(str(RESULTS / "ibm_*_n*_p*.json"))):
        d = json.load(open(f))
        rows.append({"fuente": f"Hardware ({d['backend']})", "p": d["p"],
                     **metrics.summarize(Eh, ibm.counts_to_prob(d["counts"], n), n, k)})
    table = pd.DataFrame(rows)


    e_mean = Eh.mean()
    ideal_e = {r.p: r.energía_esperada for r in table.itertuples() if r.fuente == "Ideal"}
    table["lambda_efectivo"] = [
        (r.energía_esperada - ideal_e[r.p]) / (e_mean - ideal_e[r.p]) if r.fuente not in ("Ideal", "Azar uniforme") else np.nan
        for r in table.itertuples()]
    display(table.round(4))
    table.to_csv(RESULTS / "04_hw_comparison.csv", index=False)


In [ ]:
if prob_file.exists():
    t = table[table.fuente != "Azar uniforme"]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for fuente, g in t.groupby("fuente"):
        ax[0].plot(g["p"], g["razón_aprox"], "o-", label=fuente)
        ax[1].plot(g["p"], g["prob_factible"], "o-", label=fuente)
    ax[0].axhline(0, color="gray", ls="--", lw=.8); ax[0].set(xlabel="p", ylabel="razón de aproximación", title="Calidad"); ax[0].legend()
    ax[1].axhline(metrics.feasible_mask(n, k).mean(), color="gray", ls="--", lw=.8)
    ax[1].set(xlabel="p", ylabel="P(solución factible)", title="Factibilidad (línea gris = azar)")
    plt.tight_layout(); plt.savefig(FIGS / "04_hardware_vs_ideal.png", dpi=150); plt.show()
